# XAI Lengkap -- 09_noise_robust (Tuned) -- Coffee Bean Quality Detection
### Kandidat produksi terdepan sekarang mendapat pemeriksaan paling ketat, bukan paling longgar

**Konteks:** `CBQD - HPO Seed Stability.ipynb` (5 seed x 3 model, split &amp; hyperparameter
tetap) menemukan `09_noise_robust` (mean test macro-F1 0,9540 +/- 0,0171) dan `08_multitask`
(0,9532 +/- 0,0198) secara statistik SETARA -- beda cuma 0,08pp, jauh di bawah 1 sigma
keduanya -- tapi `09_noise_robust` lebih stabil (std lebih kecil) dan arsitekturnya lebih
sederhana (single-task vs 2-head). Klaim "08_multitask menang +3pp" dari
`CBQD - HPO Optuna.ipynb` sebelumnya sudah terbukti tidak reproducible
(`CBQD - XAI HPO-Tuned.ipynb`).

**Masalahnya:** waktu itu `08_multitask` dapat baterai XAI PALING LENGKAP (Grad-CAM per head,
TCAV, causal probe, Hipotesis #2/#3) karena dialah yang paling dicurigai. `09_noise_robust`
cuma dapat "spot-check" (agreement rate + Hipotesis #2/#3, tanpa TCAV/causal-probe/embedding-
lineage) karena metriknya waktu itu terlihat paling tidak berubah dari HPO. Sekarang posisinya
terbalik -- `09_noise_robust` calon model produksi, jadi dia yang seharusnya diperiksa paling
ketat, bukan paling longgar.

**Cakupan notebook ini** (identik cakupan Family A di `CBQD - XAI.ipynb`, karena
`09_noise_robust` adalah CNN single-head flat 4-kelas, BUKAN sub-model multi-task seperti
`08_multitask`):
- Grad-CAM + korelasi centroid-vs-posisi
- TCAV 4 konsep (posisi, bentuk, warna, ukuran -- Hipotesis #1/#4/#5)
- Causal probe (reposisi, occlusion, color-jitter)
- **Embedding lineage** (NN label agreement) -- TIDAK diberikan ke `08_multitask` dulu (alasan
  arsitektural Family D tidak berlaku di sini), jadi `09_noise_robust` justru dapat cakupan
  yang LEBIH lengkap
- Hipotesis #2 (damage-leak) &amp; Hipotesis #3 (generalisasi `real_world/`), dibandingkan ke
  checkpoint lama (pre-HPO)

Checkpoint yang diperiksa: `models/checkpoints/09_noise_robust_tuned.pt` -- SUDAH ada di
DVC/R2 (hasil retrain `CBQD - XAI HPO-Tuned.ipynb`, test macro-F1=0,9739), dimuat langsung
lewat `dvc pull`, TIDAK diretrain ulang di sini.


## Section 1 -- Environment & Data Provenance Setup

In [ ]:
# Sub-Step 1.1
# Tujuan: Install dependency tambahan (tanpa menyentuh torch/torchvision)
# Catatan: hanya `captum` -- notebook ini cuma menyentuh 09_noise_robust.

!pip install -q captum

import os, json
from pathlib import Path

GIT_REPO_URL = "https://github.com/Ardiyanto24/coffee-bean-quality-detection.git"
PROJECT_DIR = "/kaggle/working/coffee-bean-quality-detection"
if not os.path.exists(PROJECT_DIR):
    os.system(f"git clone {GIT_REPO_URL} {PROJECT_DIR}")
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())


In [ ]:
# Sub-Step 1.2
# Tujuan: Konfigurasi kredensial R2 (private dataset jika ada, fallback ke Kaggle Secrets)

_matches = list(Path("/kaggle/input").rglob("r2_credentials.json")) if os.path.exists("/kaggle/input") else []
cred_path = _matches[0] if _matches else None

if cred_path is not None:
    creds = json.loads(cred_path.read_text())
    os.environ["AWS_ACCESS_KEY_ID"] = creds["R2_ACCESS_KEY_ID"]
    os.environ["AWS_SECRET_ACCESS_KEY"] = creds["R2_SECRET_ACCESS_KEY"]
    print("Kredensial R2 dimuat dari private Kaggle Dataset (nilai tidak di-print).")
else:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["AWS_ACCESS_KEY_ID"] = secrets.get_secret("R2_ACCESS_KEY_ID")
    os.environ["AWS_SECRET_ACCESS_KEY"] = secrets.get_secret("R2_SECRET_ACCESS_KEY")
    print("Kredensial R2 dimuat dari Kaggle Secrets (nilai tidak di-print).")


In [ ]:
# Sub-Step 1.3
# Tujuan: Tarik dataset + manifest + checkpoint (lama & tuned) dari R2

!pip install -q "dvc[s3]"
!dvc pull -v
print("dataset/ ada:", Path("dataset").exists())
print("dataset_preprocessed/ ada:", Path("dataset_preprocessed").exists())
print("manifest_preprocessed.csv ada:", Path("metadata/manifest_preprocessed.csv").exists())

CKPT_DIR = Path("models/checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
tuned_ckpt = CKPT_DIR / "09_noise_robust_tuned.pt"
baseline_ckpt = CKPT_DIR / "09_noise_robust.pt"
print(f"09_noise_robust_tuned.pt ada: {tuned_ckpt.exists()}")
print(f"09_noise_robust.pt (baseline pre-HPO) ada: {baseline_ckpt.exists()}")


## Section 2 -- Konfigurasi

In [ ]:
# Sub-Step 2.1
# Tujuan: Flag DRY_RUN + seed + parameter dasar

import random
import numpy as np
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DRY_RUN = False  # <-- dry-run (v1) sudah diverifikasi bersih di Kaggle (COMPLETE, checkpoint termuat benar), full run.

if DRY_RUN:
    EPOCHS_PHASE1 = 1
    EPOCHS_PHASE2 = 4
    EARLY_STOP_PATIENCE = 2
    N_AGG_PER_CLASS = 3
    N_QUAL = 4
else:
    EPOCHS_PHASE1 = 5
    EPOCHS_PHASE2 = 45
    EARLY_STOP_PATIENCE = 10
    N_AGG_PER_CLASS = 999999  # efektif: pakai semua test set (231 gambar)
    N_QUAL = 8

IMG_SIZE = 224
CLASS_NAMES = ["defect", "longberry", "peaberry", "premium"]
LABEL_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DRY_RUN={DRY_RUN} | device={device} | N_AGG_PER_CLASS={N_AGG_PER_CLASS}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print("GPU:", gpu_name)
    if "P100" in gpu_name:
        raise RuntimeError(
            f"GPU allocated is {gpu_name}, incompatible with the preinstalled PyTorch "
            "build (no Pascal/sm_60 kernels). Re-push the kernel with "
            "kernel-metadata.json machine_shape=NvidiaTeslaT4 to force a T4 allocation."
        )


In [ ]:
# Sub-Step 2.2
# Tujuan: Muat best hyperparameter 09_noise_robust dari metadata/hpo_final_summary.csv
# (dipakai HANYA kalau checkpoint tuned belum ada dan perlu retrain fallback)

import pandas as pd

hpo_summary = pd.read_csv("metadata/hpo_final_summary.csv").set_index("model")
_row = hpo_summary.loc["09_noise_robust"]
bp_noise_robust = {
    "lr_phase1": float(_row["param_lr_phase1"]), "lr_phase2": float(_row["param_lr_phase2"]),
    "weight_decay": float(_row["param_weight_decay"]), "batch_size": int(_row["param_batch_size"]),
    "scheduler_factor": float(_row["param_scheduler_factor"]), "scheduler_patience": int(_row["param_scheduler_patience"]),
    "label_smoothing": float(_row["param_label_smoothing"]), "mislabel_weight": float(_row["param_mislabel_weight"]),
    "mistake_threshold": float(_row["param_mistake_threshold"]),
}
print("[09_noise_robust] best params:", bp_noise_robust)


## Section 3 -- Data: Manifest, Dataset, Transform, DataLoader

In [ ]:
# Sub-Step 3.1
# Tujuan: Load manifest, definisikan fit/val/test/real_world DataFrame

manifest = pd.read_csv("metadata/manifest_preprocessed.csv")
PREP_DIR = Path("dataset_preprocessed")
RAW_DIR = Path("dataset")

train_pool = manifest[manifest["split"] == "train"].reset_index(drop=True)
test_df = manifest[manifest["split"] == "test"].reset_index(drop=True)
real_world_df = manifest[manifest["split"] == "real_world"].reset_index(drop=True)

fit_df = train_pool[train_pool["cv_fold"].isin([1, 2, 3])].reset_index(drop=True)
val_df = train_pool[train_pool["cv_fold"] == 0].reset_index(drop=True)

print(f"fit={len(fit_df)}  val={len(val_df)}  test={len(test_df)}  real_world={len(real_world_df)}")


In [ ]:
# Sub-Step 3.2
# Tujuan: Dataset & transform, DataLoader

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
EVAL_BATCH_SIZE = 32

train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.RandomRotation(degrees=180),
    T.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.02),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class BeanDataset(Dataset):
    def __init__(self, df, root_dir, transform, weights=None, label_fn=None):
        self.paths = [root_dir / p for p in df["image_path"]]
        label_fn = label_fn if label_fn is not None else (lambda l: LABEL_TO_IDX[l])
        self.labels = [label_fn(l) for l in df["label"]]
        self.transform = transform
        self.weights = weights if weights is not None else [1.0] * len(df)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        img = self.transform(img)
        return img, self.labels[idx], self.weights[idx]


def make_loaders(fit_kwargs, val_kwargs, batch_size):
    fit_loader = DataLoader(BeanDataset(fit_df, PREP_DIR, train_transform, **fit_kwargs),
                             batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(BeanDataset(val_df, PREP_DIR, eval_transform, **val_kwargs),
                             batch_size=batch_size, shuffle=False, num_workers=2)
    return fit_loader, val_loader


test_loader = DataLoader(BeanDataset(test_df, PREP_DIR, eval_transform),
                          batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=2)
real_world_loader = DataLoader(
    BeanDataset(real_world_df, PREP_DIR, eval_transform, label_fn=lambda l: 0),
    batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=2,
)  # label_fn dummy -- real_world tidak punya ground truth
print("DataLoaders siap.")


## Section 4 -- Fungsi Utilitas Model

In [ ]:
# Sub-Step 4.1
# Tujuan: evaluate()

import torch.nn as nn
from sklearn.metrics import f1_score, accuracy_score, classification_report

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    for images, labels, _ in loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    acc = accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=CLASS_NAMES,
                                    output_dict=True, zero_division=0)
    return {"macro_f1": macro_f1, "accuracy": acc, "report": report}


In [ ]:
# Sub-Step 4.2
# Tujuan: set_backbone_frozen() + train_one_model_hpo() -- fallback kalau checkpoint tuned
# belum ada dan perlu retrain (harusnya tidak terpakai -- checkpoint sudah ada di DVC)

import copy


def set_backbone_frozen(model, head_module, frozen: bool):
    head_param_ids = set(id(p) for p in head_module.parameters())
    for p in model.parameters():
        p.requires_grad = (id(p) in head_param_ids) or (not frozen)


def _train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    for images, labels, weights in loader:
        images, labels, weights = images.to(device), labels.to(device), weights.to(device).float()
        optimizer.zero_grad()
        outputs = model(images)
        per_sample_loss = criterion(outputs, labels)
        loss = (per_sample_loss * weights).mean() if per_sample_loss.dim() > 0 else per_sample_loss
        loss.backward()
        optimizer.step()


def train_one_model_hpo(model, head_module, fit_loader, val_loader, device, model_name,
                         lr_phase1, lr_phase2, weight_decay, scheduler_factor, scheduler_patience,
                         epochs_phase2, criterion=None):
    model = model.to(device)
    criterion = criterion if criterion is not None else nn.CrossEntropyLoss(reduction="none")
    best_state, best_val_f1, patience_counter = None, -1.0, 0

    set_backbone_frozen(model, head_module, frozen=True)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr_phase1, weight_decay=weight_decay
    )
    for epoch in range(EPOCHS_PHASE1):
        _train_epoch(model, fit_loader, optimizer, criterion, device)
        val_metrics = evaluate(model, val_loader, device)
        if val_metrics["macro_f1"] > best_val_f1:
            best_val_f1, best_state, patience_counter = val_metrics["macro_f1"], copy.deepcopy(model.state_dict()), 0
        else:
            patience_counter += 1

    set_backbone_frozen(model, head_module, frozen=False)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr_phase2, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=scheduler_factor, patience=scheduler_patience)
    patience_counter = 0
    for epoch in range(epochs_phase2):
        _train_epoch(model, fit_loader, optimizer, criterion, device)
        val_metrics = evaluate(model, val_loader, device)
        scheduler.step(val_metrics["macro_f1"])
        if val_metrics["macro_f1"] > best_val_f1:
            best_val_f1, best_state, patience_counter = val_metrics["macro_f1"], copy.deepcopy(model.state_dict()), 0
        else:
            patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_val_f1


In [ ]:
# Sub-Step 4.3
# Tujuan: build_model() -- HANYA efficientnet_b0 (satu-satunya arsitektur 09_noise_robust)

from torchvision import models as tv_models


def build_model(arch, num_classes):
    if arch == "efficientnet_b0":
        m = tv_models.efficientnet_b0(weights=tv_models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        in_f = m.classifier[-1].in_features
        m.classifier[-1] = nn.Linear(in_f, num_classes)
        head = m.classifier[-1]
    else:
        raise ValueError(f"Arsitektur tidak dikenal: {arch}")
    return m, head


In [ ]:
# Sub-Step 4.4
# Tujuan: handcrafted_features()/build_feature_matrix() -- dibutuhkan utk mistake_score
# (fallback retrain) & concept split TCAV (Sub-Step 6.5)

import cv2


def handcrafted_features(path):
    bgr = cv2.imread(str(path))
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)

    gray_f = gray.astype(np.float32)
    thresh = gray_f.mean() - 0.6 * gray_f.std()
    mask = gray_f < thresh
    h, w = gray.shape
    if mask.sum() > 0:
        ys, xs = np.where(mask)
        area_frac = mask.sum() / (h * w)
        bbox_h, bbox_w = float(ys.max() - ys.min()), float(xs.max() - xs.min())
        bbox_ratio = max(bbox_h, bbox_w) / max(min(bbox_h, bbox_w), 1e-6)
        cy, cx = ys.mean(), xs.mean()
        center_offset = float(np.hypot(cy - h / 2, cx - w / 2) / (h / 2))
        bbox = (int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max()))
    else:
        area_frac = bbox_ratio = center_offset = np.nan
        bbox = (0, 0, w, h)

    edges = cv2.Canny(gray, 100, 200)
    feats = {
        "mean_r": rgb[:, :, 0].mean(), "mean_g": rgb[:, :, 1].mean(), "mean_b": rgb[:, :, 2].mean(),
        "std_r": rgb[:, :, 0].std(), "std_g": rgb[:, :, 1].std(), "std_b": rgb[:, :, 2].std(),
        "edge_density": edges.mean() / 255, "variance": float(np.var(gray)),
        "area_frac": area_frac, "bbox_ratio": bbox_ratio, "center_offset": center_offset,
    }
    return feats, bbox


FEATURE_COLS = ["mean_r", "mean_g", "mean_b", "std_r", "std_g", "std_b",
                "edge_density", "variance", "area_frac", "bbox_ratio", "center_offset"]


def build_feature_matrix(df):
    rows = [handcrafted_features(RAW_DIR / p)[0] for p in df["orig_path"]]
    X = pd.DataFrame(rows)[FEATURE_COLS].values
    y = np.array([LABEL_TO_IDX[l] for l in df["label"]])
    return X, y


## Section 5 -- Muat Checkpoint Tuned (atau Retrain Fallback)

`09_noise_robust_tuned.pt` seharusnya SUDAH ada setelah `dvc pull` (hasil retrain
`CBQD - XAI HPO-Tuned.ipynb`, test macro-F1=0,9739 pada eksperimen itu). Kalau karena
sebab apa pun file itu tidak ada, retrain fallback dengan hyperparameter yang sama.

In [ ]:
# Sub-Step 5.1
# Tujuan: get_or_train_tuned() -- muat checkpoint yang sudah ada, retrain hanya kalau perlu

def get_or_train_tuned():
    path = CKPT_DIR / "09_noise_robust_tuned.pt"
    model, head = build_model("efficientnet_b0", 4)
    if path.exists():
        model.load_state_dict(torch.load(path, map_location=device))
        print(f"[09_noise_robust_tuned] checkpoint dimuat dari {path}")
        return model.to(device).eval()

    print(f"[09_noise_robust_tuned] checkpoint TIDAK ditemukan -- retrain fallback dari best_params")
    from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
    from sklearn.ensemble import RandomForestClassifier

    X_fit_m9, y_fit_m9 = build_feature_matrix(fit_df)
    sgkf = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED)
    rf = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)
    proba_oof = cross_val_predict(rf, X_fit_m9, y_fit_m9, cv=sgkf,
                                   groups=fit_df["cluster_id"].values, method="predict_proba")
    true_proba = proba_oof[np.arange(len(y_fit_m9)), y_fit_m9]
    max_proba = proba_oof.max(axis=1)
    mistake_score = max_proba - true_proba
    flagged_mask = mistake_score > bp_noise_robust["mistake_threshold"]
    sample_weights_fit = np.where(flagged_mask, bp_noise_robust["mislabel_weight"], 1.0)

    fit_loader, val_loader = make_loaders({"weights": sample_weights_fit.tolist()}, {}, bp_noise_robust["batch_size"])
    criterion = nn.CrossEntropyLoss(label_smoothing=bp_noise_robust["label_smoothing"], reduction="none")
    model, val_f1 = train_one_model_hpo(
        model, head, fit_loader, val_loader, device, "09_noise_robust_tuned",
        lr_phase1=bp_noise_robust["lr_phase1"], lr_phase2=bp_noise_robust["lr_phase2"],
        weight_decay=bp_noise_robust["weight_decay"], scheduler_factor=bp_noise_robust["scheduler_factor"],
        scheduler_patience=bp_noise_robust["scheduler_patience"], epochs_phase2=EPOCHS_PHASE2, criterion=criterion,
    )
    torch.save(model.state_dict(), path)
    print(f"[09_noise_robust_tuned] selesai retrain fallback, val_macro_f1={val_f1:.4f}")
    return model.eval()


model_tuned = get_or_train_tuned()
torch.cuda.empty_cache()


In [ ]:
# Sub-Step 5.2
# Tujuan: Sanity check -- verifikasi test macro-F1 checkpoint yang dimuat

test_metrics_tuned = evaluate(model_tuned, test_loader, device)
print(f"[09_noise_robust tuned] test_macro_f1={test_metrics_tuned['macro_f1']:.4f}  "
      f"test_accuracy={test_metrics_tuned['accuracy']:.4f}")
for cls in CLASS_NAMES:
    print(f"  recall {cls}: {test_metrics_tuned['report'][cls]['recall']:.4f}")


## Section 6 -- Fungsi Utilitas XAI (identik `CBQD - XAI.ipynb` Family A)

In [ ]:
# Sub-Step 6.1
# Tujuan: Grad-CAM (Captum) + IG + Occlusion + centroid

from captum.attr import LayerGradCam, LayerAttribution, IntegratedGradients, Occlusion


def gradcam_heatmaps(model, layer, images_tensor, target_classes):
    lgc = LayerGradCam(model, layer)
    heatmaps = []
    for i in range(images_tensor.shape[0]):
        a = lgc.attribute(images_tensor[i:i + 1].to(device), target=int(target_classes[i]))
        a = LayerAttribution.interpolate(a, (IMG_SIZE, IMG_SIZE))
        heatmaps.append(a.squeeze().detach().cpu().numpy())
    return heatmaps


def ig_heatmaps(model, images_tensor, target_classes, n_steps=20):
    ig = IntegratedGradients(model)
    heatmaps = []
    for i in range(images_tensor.shape[0]):
        a = ig.attribute(images_tensor[i:i + 1].to(device), target=int(target_classes[i]), n_steps=n_steps)
        heatmaps.append(a.squeeze().abs().sum(dim=0).detach().cpu().numpy())
    return heatmaps


def occlusion_heatmaps(model, images_tensor, target_classes, window=32, stride=16):
    occ = Occlusion(model)
    heatmaps = []
    for i in range(images_tensor.shape[0]):
        a = occ.attribute(images_tensor[i:i + 1].to(device), target=int(target_classes[i]),
                           sliding_window_shapes=(3, window, window), strides=(3, stride, stride))
        heatmaps.append(a.squeeze().abs().sum(dim=0).detach().cpu().numpy())
    return heatmaps


def heatmap_centroid_offset(heatmap):
    h, w = heatmap.shape
    hm = np.clip(heatmap, 0, None)
    if hm.sum() <= 1e-8:
        return np.nan
    ys, xs = np.indices((h, w))
    cy = (ys * hm).sum() / hm.sum()
    cx = (xs * hm).sum() / hm.sum()
    return float(np.hypot(cy - h / 2, cx - w / 2) / (h / 2))


In [ ]:
# Sub-Step 6.2
# Tujuan: TCAV manual (layer activation + CAV + directional derivative)

from sklearn.linear_model import LogisticRegression
from scipy.stats import ttest_ind

GRAD_BATCH_SIZE = 32


def layer_activation_batch(model, layer, images_tensor):
    acts = {}
    def hook(m, i, o): acts["v"] = o.detach()
    h = layer.register_forward_hook(hook)
    outs = []
    with torch.no_grad():
        for i in range(0, images_tensor.shape[0], GRAD_BATCH_SIZE):
            model(images_tensor[i:i + GRAD_BATCH_SIZE].to(device))
            a = acts["v"]
            a = a.mean(dim=[2, 3]) if a.dim() == 4 else (a.mean(dim=1) if a.dim() == 3 else a)
            outs.append(a.cpu().numpy())
    h.remove()
    torch.cuda.empty_cache()
    return np.concatenate(outs, axis=0)


def layer_grad_wrt_target(model, layer, images_tensor, target_class):
    acts = {}
    def hook(m, i, o):
        o.retain_grad()
        acts["v"] = o
    h = layer.register_forward_hook(hook)
    outs = []
    for i in range(0, images_tensor.shape[0], GRAD_BATCH_SIZE):
        batch = images_tensor[i:i + GRAD_BATCH_SIZE].to(device)
        out = model(batch)
        logit = out[:, target_class].sum()
        model.zero_grad(set_to_none=True)
        logit.backward()
        a = acts["v"]
        grad = a.grad
        grad = grad.mean(dim=[2, 3]) if grad.dim() == 4 else (grad.mean(dim=1) if grad.dim() == 3 else grad)
        outs.append(grad.detach().cpu().numpy())
    h.remove()
    torch.cuda.empty_cache()
    return np.concatenate(outs, axis=0)


def compute_cav(pos_acts, neg_acts, seed=SEED):
    X = np.concatenate([pos_acts, neg_acts], axis=0)
    y = np.concatenate([np.ones(len(pos_acts)), np.zeros(len(neg_acts))])
    clf = LogisticRegression(max_iter=1000, random_state=seed).fit(X, y)
    cav = clf.coef_[0]
    return cav / (np.linalg.norm(cav) + 1e-8)


def tcav_score_with_significance(model, layer, concept_pos_imgs, concept_neg_imgs,
                                  target_imgs, target_class, n_random=5):
    pos_acts = layer_activation_batch(model, layer, concept_pos_imgs)
    neg_acts = layer_activation_batch(model, layer, concept_neg_imgs)
    cav = compute_cav(pos_acts, neg_acts)
    grads = layer_grad_wrt_target(model, layer, target_imgs, target_class)
    sensitivities = grads @ cav
    real_score = float((sensitivities > 0).mean())

    pool_acts = np.concatenate([pos_acts, neg_acts], axis=0)
    n_pos = len(pos_acts)
    rng = np.random.RandomState(SEED)
    random_scores = []
    for _ in range(n_random):
        perm = rng.permutation(len(pool_acts))
        rand_cav = compute_cav(pool_acts[perm[:n_pos]], pool_acts[perm[n_pos:]])
        rand_sens = grads @ rand_cav
        random_scores.append(float((rand_sens > 0).mean()))
    _, p_value = ttest_ind([real_score], random_scores) if len(set(random_scores)) > 1 else (np.nan, np.nan)
    return {"tcav_score": real_score, "random_scores_mean": float(np.mean(random_scores)),
            "p_value": float(p_value) if p_value == p_value else None}


In [ ]:
# Sub-Step 6.3
# Tujuan: Probe kausal: reposisi, occlusion, color-jitter

def probe_reposition(orig_path, shift_frac=0.25, margin=0.2):
    _, bbox = handcrafted_features(orig_path)
    img = cv2.cvtColor(cv2.imread(str(orig_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    x0, y0, x1, y1 = bbox
    bw, bh = max(x1 - x0, 1), max(y1 - y0, 1)
    half = max(bw, bh) * (1 + margin) / 2

    def crop_at(cx, cy):
        xa, ya = int(max(cx - half, 0)), int(max(cy - half, 0))
        xb, yb = int(min(cx + half, w)), int(min(cy + half, h))
        if xb <= xa or yb <= ya:
            return Image.fromarray(img).resize((IMG_SIZE, IMG_SIZE))
        return Image.fromarray(img[ya:yb, xa:xb]).resize((IMG_SIZE, IMG_SIZE))

    cx0, cy0 = (x0 + x1) / 2, (y0 + y1) / 2
    return crop_at(cx0, cy0), crop_at(cx0 + shift_frac * w, cy0)


def probe_occlude(pil_img, region="center"):
    arr = np.array(pil_img).copy()
    h, w = arr.shape[:2]
    fill = arr.reshape(-1, arr.shape[-1]).mean(axis=0).astype(arr.dtype)
    slices = {
        "center": (slice(h // 4, 3 * h // 4), slice(w // 4, 3 * w // 4)),
        "left": (slice(0, h), slice(0, w // 2)),
        "right": (slice(0, h), slice(w // 2, w)),
    }[region]
    arr[slices] = fill
    return Image.fromarray(arr)


def probe_color_jitter(pil_img, brightness_factor=1.3, saturation_factor=1.3):
    from PIL import ImageEnhance
    img = ImageEnhance.Brightness(pil_img).enhance(brightness_factor)
    img = ImageEnhance.Color(img).enhance(saturation_factor)
    return img


def prob_delta(predict_fn, pil_a, pil_b, class_idx):
    pa = predict_fn(pil_a)[class_idx]
    pb = predict_fn(pil_b)[class_idx]
    return float(abs(pa - pb)), float(pa), float(pb)


In [ ]:
# Sub-Step 6.4
# Tujuan: correlate_safe() + sample_per_class() + load_pil_batch() + predict_proba_pil()

from scipy.stats import pearsonr


def correlate_safe(values, meta_values):
    values = np.asarray(values, dtype=float)
    meta_values = np.asarray(meta_values, dtype=float)
    valid = ~(np.isnan(values) | np.isnan(meta_values))
    if valid.sum() < 3 or np.std(values[valid]) < 1e-8 or np.std(meta_values[valid]) < 1e-8:
        return np.nan, np.nan
    r, p = pearsonr(values[valid], meta_values[valid])
    return float(r), float(p)


def sample_per_class(df, n_per_class, seed=SEED):
    parts = []
    for cls in CLASS_NAMES:
        sub_df = df[df["label"] == cls]
        n = min(n_per_class, len(sub_df))
        parts.append(sub_df.sample(n=n, random_state=seed))
    return pd.concat(parts).reset_index(drop=True)


def load_pil_batch(paths, root_dir, transform):
    imgs = [transform(Image.open(root_dir / p if not str(p).startswith(str(root_dir)) else p).convert("RGB"))
            for p in paths]
    return torch.stack(imgs)


def predict_proba_pil(model, pil_img):
    x = eval_transform(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        return torch.softmax(model(x), dim=1).cpu().numpy()[0]


In [ ]:
# Sub-Step 6.5
# Tujuan: Concept split untuk TCAV (Hipotesis #1/#4/#5) -- dihitung dari dataset_preprocessed

def compute_features_for_paths(paths):
    return pd.DataFrame([handcrafted_features(p)[0] for p in paths])


test_paths_prep = [PREP_DIR / p for p in test_df["image_path"]]
test_feats_prep = compute_features_for_paths(test_paths_prep)
test_feats_prep["image_path"] = test_df["image_path"].values
test_feats_prep["label"] = test_df["label"].values


def build_concept_examples(feature_col, concept_is_low, quantile=0.3, n=20):
    vals = test_feats_prep[feature_col].values
    valid = ~np.isnan(vals)
    sub = test_feats_prep[valid].copy()
    lo_thr, hi_thr = sub[feature_col].quantile(quantile), sub[feature_col].quantile(1 - quantile)
    pos_df = sub[sub[feature_col] <= lo_thr] if concept_is_low else sub[sub[feature_col] >= hi_thr]
    neg_df = sub[sub[feature_col] >= hi_thr] if concept_is_low else sub[sub[feature_col] <= lo_thr]
    pos_df = pos_df.sample(n=min(n, len(pos_df)), random_state=SEED)
    neg_df = neg_df.sample(n=min(n, len(neg_df)), random_state=SEED)
    return pos_df["image_path"].tolist(), neg_df["image_path"].tolist()


CONCEPT_DEFS = [
    {"name": "off_center", "feature": "center_offset", "concept_is_low": True, "target_class": "premium"},
    {"name": "elongated_shape", "feature": "bbox_ratio", "concept_is_low": False, "target_class": "longberry"},
    {"name": "dark_color", "feature": "mean_r", "concept_is_low": True, "target_class": "defect"},
    {"name": "large_area", "feature": "area_frac", "concept_is_low": False, "target_class": "peaberry"},
]
for cdef in CONCEPT_DEFS:
    pos_paths, neg_paths = build_concept_examples(cdef["feature"], cdef["concept_is_low"])
    cdef["pos_paths"] = pos_paths
    cdef["neg_paths"] = neg_paths
    print(f"Konsep '{cdef['name']}': {len(pos_paths)} positif, {len(neg_paths)} negatif")


In [ ]:
# Sub-Step 6.6
# Tujuan: Embedding lineage -- extract_embeddings_for_df() + nn_label_agreement()
# (TIDAK diberikan ke 08_multitask sebelumnya karena alasan arsitektural Family D --
# 09_noise_robust CNN plain, jadi cakupannya justru LEBIH lengkap di sini)

from sklearn.neighbors import NearestNeighbors


def extract_embeddings_for_df(model, layer, df, root_dir, transform, batch_size=32):
    loader = DataLoader(BeanDataset(df, root_dir, transform), batch_size=batch_size, shuffle=False, num_workers=2)
    embs = []
    for images, _, _ in loader:
        embs.append(layer_activation_batch(model, layer, images))
    return np.concatenate(embs, axis=0)


def nn_label_agreement(train_embs, train_labels, query_embs, query_labels, k=5):
    nn = NearestNeighbors(n_neighbors=k).fit(train_embs)
    _, idx = nn.kneighbors(query_embs)
    train_labels = np.asarray(train_labels)
    agree = np.array([(train_labels[row] == query_labels[i]).mean() for i, row in enumerate(idx)])
    return agree


In [ ]:
# Sub-Step 6.7
# Tujuan: FIG_DIR + heatmap overlay + predict_proba_loader + summarize_hypothesis3 + damage_leak_test

import matplotlib
matplotlib.use("Agg")
import matplotlib.cm as cm
import matplotlib.pyplot as plt

FIG_DIR = Path("results/xai_09_noise_robust_figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)
N_VIZ_PER_CLASS = 1


def heatmap_overlay_rgb(pil_img, heatmap, alpha=0.5):
    img_arr = np.array(pil_img.resize((IMG_SIZE, IMG_SIZE)).convert("RGB")) / 255.0
    hm = heatmap - np.nanmin(heatmap)
    hm = hm / (np.nanmax(hm) + 1e-8)
    heat_colored = cm.jet(hm)[:, :, :3]
    overlay = (1 - alpha) * img_arr + alpha * heat_colored
    return np.clip(overlay, 0, 1)


@torch.no_grad()
def predict_proba_loader(model, loader):
    model.eval()
    all_proba = []
    for images, _, _ in loader:
        all_proba.append(torch.softmax(model(images.to(device)), dim=1).cpu().numpy())
    return np.concatenate(all_proba, axis=0)


def summarize_hypothesis3(name, test_proba, rw_proba):
    test_conf, rw_conf = test_proba.max(axis=1), rw_proba.max(axis=1)
    rw_pred = rw_proba.argmax(axis=1)
    rw_class_dist = {cname: float((rw_pred == i).mean()) for i, cname in enumerate(CLASS_NAMES)}
    print(f"--- {name} --- Confidence test: {test_conf.mean():.3f} | real_world: {rw_conf.mean():.3f}")
    print(f"  Distribusi prediksi real_world: {rw_class_dist}")
    return {"model": name, "test_confidence_mean": float(test_conf.mean()),
            "real_world_confidence_mean": float(rw_conf.mean()),
            "real_world_class_distribution": rw_class_dist}


def damage_leak_test(model, layer, model_name, defect_idx):
    defect_paths = test_df[test_df["label"] == "defect"]["image_path"].tolist()
    nondefect_df = test_df[test_df["label"] != "defect"]
    nondefect_paths_all = nondefect_df["image_path"].tolist()

    rng = np.random.RandomState(SEED)
    n_concept = min(30, len(defect_paths), len(nondefect_paths_all))
    defect_sample = rng.choice(defect_paths, n_concept, replace=False)
    nondefect_sample_for_cav = rng.choice(nondefect_paths_all, n_concept, replace=False)

    defect_imgs = load_pil_batch(list(defect_sample), PREP_DIR, eval_transform)
    nondefect_imgs_cav = load_pil_batch(list(nondefect_sample_for_cav), PREP_DIR, eval_transform)
    damage_cav = compute_cav(
        layer_activation_batch(model, layer, defect_imgs),
        layer_activation_batch(model, layer, nondefect_imgs_cav),
    )

    eval_df = sample_per_class(nondefect_df, N_AGG_PER_CLASS)
    eval_imgs = load_pil_batch(eval_df["image_path"].tolist(), PREP_DIR, eval_transform)
    grads = layer_grad_wrt_target(model, layer, eval_imgs, defect_idx)
    sensitivities = grads @ damage_cav

    predict_fn = lambda img: predict_proba_pil(model, img)
    predicted_idx = np.array([
        predict_fn(Image.open(PREP_DIR / p).convert("RGB").resize((IMG_SIZE, IMG_SIZE))).argmax()
        for p in eval_df["image_path"]
    ])
    misclassified_as_defect = (predicted_idx == defect_idx).astype(float)

    leak_score = float((sensitivities > 0).mean())
    r, p = correlate_safe(sensitivities, misclassified_as_defect)
    n_misclassified = int(misclassified_as_defect.sum())
    print(f"[{model_name}] Damage-leak score: {leak_score:.3f} | {n_misclassified}/{len(eval_df)} "
          f"non-defect justru diprediksi defect | r={r}, p={p}")
    return {"model": model_name, "damage_leak_score": leak_score,
            "n_misclassified_as_defect": n_misclassified, "n_eval": len(eval_df),
            "damage_leak_vs_misclass_r": r, "damage_leak_vs_misclass_p": p}


In [ ]:
# Sub-Step 6.8
# Tujuan: Muat baseline lama (pre-HPO, dari CBQD - XAI.ipynb) untuk perbandingan

old_hyp2 = pd.read_csv("metadata/xai_hypothesis2_damage_leak.csv").set_index("model")
old_hyp3 = pd.read_csv("metadata/xai_hypothesis3_real_world.csv").set_index("model")
old_summary = pd.read_csv("metadata/xai_summary.csv").set_index("model")
print("Baseline lama (pre-HPO) 09_noise_robust dimuat untuk perbandingan.")


## Section 7 -- Baterai Lengkap: 09_noise_robust (Tuned)

Identik cakupan Family A `CBQD - XAI.ipynb`: Grad-CAM + centroid, TCAV 4 konsep, causal probe,
embedding lineage, Hipotesis #2 (damage-leak), Hipotesis #3 (real-world) -- setiap hasil
dibandingkan langsung ke checkpoint LAMA (pre-HPO) yang sudah tersimpan di
`metadata/xai_summary.csv` / `xai_hypothesis2_damage_leak.csv` / `xai_hypothesis3_real_world.csv`.

In [ ]:
# Sub-Step 7.1
# Tujuan: Grad-CAM + korelasi centroid-vs-center_offset

FAMILY_A_LAYER_09 = model_tuned.features[-1]

sample_df = sample_per_class(test_df, N_AGG_PER_CLASS)
sample_labels = np.array([LABEL_TO_IDX[l] for l in sample_df["label"]])
images_tensor = load_pil_batch(sample_df["image_path"].tolist(), PREP_DIR, eval_transform)

heatmaps = gradcam_heatmaps(model_tuned, FAMILY_A_LAYER_09, images_tensor, sample_labels)
centroids = np.array([heatmap_centroid_offset(hm) for hm in heatmaps])
merged = sample_df.merge(test_feats_prep[["image_path", "center_offset"]], on="image_path", how="left")
r, p = correlate_safe(centroids, merged["center_offset"].values)

result_09 = {"model": "09_noise_robust_tuned", "centroid_offset_mean": float(np.nanmean(centroids)),
             "centroid_vs_center_offset_r": r, "centroid_vs_center_offset_p": p}
old_r = float(old_summary.loc["09_noise_robust", "centroid_vs_center_offset_r"])
print(f"[09_noise_robust tuned] Grad-CAM centroid vs center_offset: r={r}, p={p}  (baseline lama: r={old_r:.4f})")


In [ ]:
# Sub-Step 7.2
# Tujuan: TCAV 4 konsep (Hipotesis #1/#4/#5)

tcav_09 = {}
for cdef in CONCEPT_DEFS:
    pos_imgs = load_pil_batch(cdef["pos_paths"], PREP_DIR, eval_transform)
    neg_imgs = load_pil_batch(cdef["neg_paths"], PREP_DIR, eval_transform)
    target_idx = LABEL_TO_IDX[cdef["target_class"]]
    target_mask = sample_labels == target_idx
    target_imgs = images_tensor[target_mask] if target_mask.sum() >= 2 else images_tensor
    tcav_09[cdef["name"]] = tcav_score_with_significance(model_tuned, FAMILY_A_LAYER_09, pos_imgs, neg_imgs, target_imgs, target_idx)
    old_score = float(old_summary.loc["09_noise_robust", f"tcav_{cdef['name']}_score"])
    cur = tcav_09[cdef["name"]]
    print(f"[09_noise_robust tuned] tcav[{cdef['name']}]: score={cur['tcav_score']:.3f} p={cur['p_value']} "
          f"(baseline lama: score={old_score:.3f})")
result_09["tcav"] = tcav_09
torch.cuda.empty_cache()


In [ ]:
# Sub-Step 7.3
# Tujuan: Probe kausal -- reposisi, occlusion, color-jitter

predict_fn_09 = lambda img: predict_proba_pil(model_tuned, img)
reposition_deltas, occlusion_deltas, color_deltas = [], [], []
for _, row in sample_df.iterrows():
    cls_idx = LABEL_TO_IDX[row["label"]]
    try:
        crop_a, crop_b = probe_reposition(RAW_DIR / row["orig_path"])
        d, _, _ = prob_delta(predict_fn_09, crop_a, crop_b, cls_idx)
        reposition_deltas.append(d)
    except Exception:
        pass
    prep_img = Image.open(PREP_DIR / row["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    occluded = probe_occlude(prep_img, region="center")
    d2, _, _ = prob_delta(predict_fn_09, prep_img, occluded, cls_idx)
    occlusion_deltas.append(d2)
    jittered = probe_color_jitter(prep_img)
    d3, _, _ = prob_delta(predict_fn_09, prep_img, jittered, cls_idx)
    color_deltas.append(d3)

result_09["probe_reposition_mean_delta"] = float(np.mean(reposition_deltas)) if reposition_deltas else np.nan
result_09["probe_occlusion_center_mean_delta"] = float(np.mean(occlusion_deltas))
result_09["probe_color_jitter_mean_delta"] = float(np.mean(color_deltas))
print(f"[09_noise_robust tuned] reposition={result_09['probe_reposition_mean_delta']:.4f} "
      f"(lama={float(old_summary.loc['09_noise_robust','probe_reposition_mean_delta']):.4f})")
print(f"[09_noise_robust tuned] occlusion_center={result_09['probe_occlusion_center_mean_delta']:.4f} "
      f"(lama={float(old_summary.loc['09_noise_robust','probe_occlusion_center_mean_delta']):.4f})")
print(f"[09_noise_robust tuned] color_jitter={result_09['probe_color_jitter_mean_delta']:.4f} "
      f"(lama={float(old_summary.loc['09_noise_robust','probe_color_jitter_mean_delta']):.4f})")
torch.cuda.empty_cache()


In [ ]:
# Sub-Step 7.4
# Tujuan: Embedding lineage -- NN label agreement

train_embs = extract_embeddings_for_df(model_tuned, FAMILY_A_LAYER_09, fit_df, PREP_DIR, eval_transform)
train_labels_arr = np.array([LABEL_TO_IDX[l] for l in fit_df["label"]])
query_embs = extract_embeddings_for_df(model_tuned, FAMILY_A_LAYER_09, sample_df, PREP_DIR, eval_transform)
agree = nn_label_agreement(train_embs, train_labels_arr, query_embs, sample_labels, k=5)
result_09["nn_lineage_label_agreement_mean"] = float(np.mean(agree))
old_agree = float(old_summary.loc["09_noise_robust", "nn_lineage_label_agreement_mean"])
print(f"[09_noise_robust tuned] NN lineage label agreement: {result_09['nn_lineage_label_agreement_mean']:.4f} "
      f"(baseline lama: {old_agree:.4f})")


In [ ]:
# Sub-Step 7.5
# Tujuan: Hipotesis #2 (damage-leak), dibandingkan checkpoint lama

result_09_hyp2 = damage_leak_test(model_tuned, FAMILY_A_LAYER_09, "09_noise_robust_tuned", LABEL_TO_IDX["defect"])
old_score_hyp2 = float(old_hyp2.loc["09_noise_robust", "damage_leak_score"])
result_09_hyp2["damage_leak_score_baseline"] = old_score_hyp2
result_09_hyp2["damage_leak_score_delta"] = result_09_hyp2["damage_leak_score"] - old_score_hyp2
print(f"[09_noise_robust] damage_leak_score: tuned={result_09_hyp2['damage_leak_score']:.4f} vs "
      f"baseline={old_score_hyp2:.4f} (delta={result_09_hyp2['damage_leak_score_delta']:+.4f})")
torch.cuda.empty_cache()


In [ ]:
# Sub-Step 7.6
# Tujuan: Hipotesis #3 (generalisasi real_world), dibandingkan checkpoint lama

test_proba_09 = predict_proba_loader(model_tuned, test_loader)
rw_proba_09 = predict_proba_loader(model_tuned, real_world_loader)
result_09_hyp3 = summarize_hypothesis3("09_noise_robust_tuned", test_proba_09, rw_proba_09)
old_conf = float(old_hyp3.loc["09_noise_robust", "real_world_confidence_mean"])
result_09_hyp3["real_world_confidence_baseline"] = old_conf
result_09_hyp3["real_world_confidence_delta"] = result_09_hyp3["real_world_confidence_mean"] - old_conf
print(f"[09_noise_robust] real_world_confidence: tuned={result_09_hyp3['real_world_confidence_mean']:.4f} vs "
      f"baseline={old_conf:.4f} (delta={result_09_hyp3['real_world_confidence_delta']:+.4f})")


## Section 8 -- Konsolidasi & Kesimpulan

In [ ]:
# Sub-Step 8.1
# Tujuan: Gabungkan seluruh temuan jadi satu baris ringkasan, simpan CSV

Path("metadata").mkdir(exist_ok=True)

summary_row = {
    "model": "09_noise_robust", "checkpoint_test_macro_f1": test_metrics_tuned["macro_f1"],
    "centroid_vs_center_offset_r": result_09["centroid_vs_center_offset_r"],
    "centroid_vs_center_offset_r_baseline": float(old_summary.loc["09_noise_robust", "centroid_vs_center_offset_r"]),
    "probe_reposition_mean_delta": result_09["probe_reposition_mean_delta"],
    "probe_reposition_mean_delta_baseline": float(old_summary.loc["09_noise_robust", "probe_reposition_mean_delta"]),
    "probe_occlusion_center_mean_delta": result_09["probe_occlusion_center_mean_delta"],
    "probe_occlusion_center_mean_delta_baseline": float(old_summary.loc["09_noise_robust", "probe_occlusion_center_mean_delta"]),
    "probe_color_jitter_mean_delta": result_09["probe_color_jitter_mean_delta"],
    "probe_color_jitter_mean_delta_baseline": float(old_summary.loc["09_noise_robust", "probe_color_jitter_mean_delta"]),
    "nn_lineage_label_agreement_mean": result_09["nn_lineage_label_agreement_mean"],
    "nn_lineage_label_agreement_mean_baseline": float(old_summary.loc["09_noise_robust", "nn_lineage_label_agreement_mean"]),
    "damage_leak_score": result_09_hyp2["damage_leak_score"],
    "damage_leak_score_delta": result_09_hyp2["damage_leak_score_delta"],
    "real_world_confidence": result_09_hyp3["real_world_confidence_mean"],
    "real_world_confidence_delta": result_09_hyp3["real_world_confidence_delta"],
}
for cdef in CONCEPT_DEFS:
    cname = cdef["name"]
    summary_row[f"tcav_{cname}_score"] = tcav_09[cname]["tcav_score"]
    summary_row[f"tcav_{cname}_p"] = tcav_09[cname]["p_value"]
    summary_row[f"tcav_{cname}_score_baseline"] = float(old_summary.loc["09_noise_robust", f"tcav_{cname}_score"])

summary_df = pd.DataFrame([summary_row])
summary_df.to_csv("metadata/xai_09_noise_robust_full_summary.csv", index=False)

status_text = "BELUM final (sampel kecil)" if DRY_RUN else "hasil run penuh"
print(f"DRY_RUN={DRY_RUN} -- angka di bawah ini {status_text}")
print()
pd.set_option("display.max_columns", None, "display.width", 250)
print(summary_df.round(4).T.to_string())


In [ ]:
# Sub-Step 8.2
# Tujuan: Interpretasi -- apakah 09_noise_robust (tuned) aman difiksasi sebagai model produksi

_tcav_flags = [name for name in tcav_09 if tcav_09[name]["p_value"] is not None and tcav_09[name]["p_value"] < 0.05]
_leak_flag = abs(result_09_hyp2["damage_leak_score_delta"]) > 0.1
_rw_flag = result_09_hyp3["real_world_confidence_delta"] < -0.1

_leak_note = "(FLAG: >0.1)" if _leak_flag else "(wajar)"
_rw_note = "(FLAG: turun >0.1)" if _rw_flag else "(wajar)"
print(f"""
=== Ringkasan sinyal shortcut untuk 09_noise_robust (tuned) ===
- TCAV signifikan (p<0.05) pada konsep: {_tcav_flags if _tcav_flags else 'TIDAK ADA'}
- Damage-leak delta vs baseline lama: {result_09_hyp2['damage_leak_score_delta']:+.4f} {_leak_note}
- Real-world confidence delta vs baseline lama: {result_09_hyp3['real_world_confidence_delta']:+.4f} {_rw_note}
- probe_color_jitter_mean_delta: {result_09['probe_color_jitter_mean_delta']:.4f}
  (pembanding: 08_multitask tuned = 0,2400 dari CBQD - XAI HPO-Tuned.ipynb Sub-Step 7.2)

Cara membaca:
- Kalau TIDAK ADA konsep TCAV signifikan, damage-leak & real-world delta kecil (dekat 0) --
  checkpoint tuned 09_noise_robust berperilaku SAMA seperti checkpoint pre-HPO yang sudah
  lolos XAI sebelumnya. Ini bukti XAI yang setara ketatnya dengan 08_multitask dulu.
- probe_color_jitter_mean_delta yang lebih kecil dari 08_multitask (0,24) berarti
  09_noise_robust lebih robust terhadap variasi warna ekstrem -- argumen tambahan
  mendukungnya sebagai kandidat produksi dibanding 08_multitask, bukan cuma soal
  stabilitas macro-F1 (lihat CBQD - HPO Seed Stability.ipynb).
- nn_lineage_label_agreement_mean rendah (baik checkpoint lama maupun tuned) mengindikasikan
  masalah DATA (near-duplicate cross-class dari EDA v2 Section 09) -- bukan sesuatu yang bisa
  diperbaiki lewat pilihan model, dicatat sebagai batasan yang sudah diketahui.

KESIMPULAN: kalau seluruh sinyal di atas bersih, ARSITEKTUR & hyperparameter 09_noise_robust
punya bukti XAI yang cukup untuk difiksasi. Checkpoint SPESIFIK yang dipakai produksi tetap
TIDAK BOLEH hasil cherry-pick seed dengan skor tertinggi dari seed-sweep (bentuk optimism bias
yang sama seperti temuan HPO Optuna sebelumnya) -- retrain sekali lagi dengan seed baru yang
belum pernah dipakai, atau ensemble kelima checkpoint seed-sweep, sebelum benar-benar deploy.
""")


## Section 9 -- Visualisasi untuk Laporan

In [ ]:
# Sub-Step 9.1
# Tujuan: Grad-CAM + IG + Occlusion overlay -- 1 contoh per kelas

viz_df = sample_per_class(test_df, N_VIZ_PER_CLASS)
viz_imgs = load_pil_batch(viz_df["image_path"].tolist(), PREP_DIR, eval_transform)
viz_labels = np.array([LABEL_TO_IDX[l] for l in viz_df["label"]])
viz_pil_originals = [Image.open(PREP_DIR / p).convert("RGB").resize((IMG_SIZE, IMG_SIZE)) for p in viz_df["image_path"]]

gc_maps = gradcam_heatmaps(model_tuned, FAMILY_A_LAYER_09, viz_imgs, viz_labels)
ig_maps_v = ig_heatmaps(model_tuned, viz_imgs, viz_labels)
occ_maps_v = occlusion_heatmaps(model_tuned, viz_imgs, viz_labels)

n = len(viz_df)
fig, axes = plt.subplots(n, 4, figsize=(10, 2.5 * n))
if n == 1:
    axes = axes.reshape(1, -1)
for i in range(n):
    axes[i, 0].imshow(viz_pil_originals[i]); axes[i, 0].set_ylabel(viz_df.iloc[i]["label"], fontsize=10)
    axes[i, 1].imshow(heatmap_overlay_rgb(viz_pil_originals[i], gc_maps[i]))
    axes[i, 2].imshow(heatmap_overlay_rgb(viz_pil_originals[i], ig_maps_v[i]))
    axes[i, 3].imshow(heatmap_overlay_rgb(viz_pil_originals[i], occ_maps_v[i]))
    for j in range(4):
        axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
    if i == 0:
        for j, t in enumerate(["Original", "Grad-CAM", "Integrated Gradients", "Occlusion"]):
            axes[i, j].set_title(t, fontsize=10)
fig.suptitle("09_noise_robust (tuned)", fontsize=12)
fig.tight_layout()
fig.savefig(FIG_DIR / "gradcam_09_noise_robust_tuned.png", dpi=110, bbox_inches="tight")
plt.close(fig)
print(f"Disimpan: {FIG_DIR / 'gradcam_09_noise_robust_tuned.png'}")


In [ ]:
# Sub-Step 9.2
# Tujuan: Visual Grad-CAM pada contoh real_world/ -- tetap fokus ke bean?

rw_viz_df = real_world_df.sample(n=min(8, len(real_world_df)), random_state=SEED).reset_index(drop=True)
rw_viz_imgs = load_pil_batch(rw_viz_df["image_path"].tolist(), PREP_DIR, eval_transform)
rw_viz_pil = [Image.open(PREP_DIR / p).convert("RGB").resize((IMG_SIZE, IMG_SIZE)) for p in rw_viz_df["image_path"]]

rw_proba_viz = np.array([predict_proba_pil(model_tuned, img) for img in rw_viz_pil])
rw_pred_viz = rw_proba_viz.argmax(axis=1)
rw_heatmaps = gradcam_heatmaps(model_tuned, FAMILY_A_LAYER_09, rw_viz_imgs, rw_pred_viz)

fig, axes = plt.subplots(2, 4, figsize=(11, 5.4))
for i in range(len(rw_viz_pil)):
    r, c = divmod(i, 4)
    axes[r, c].imshow(heatmap_overlay_rgb(rw_viz_pil[i], rw_heatmaps[i]))
    axes[r, c].set_title(f"pred={CLASS_NAMES[rw_pred_viz[i]]}\nP={rw_proba_viz[i][rw_pred_viz[i]]:.2f}", fontsize=9)
    axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
fig.suptitle("Grad-CAM pada real_world/ (09_noise_robust tuned) -- sanity check generalisasi", fontsize=11)
fig.tight_layout()
fig.savefig(FIG_DIR / "real_world_gradcam_09_noise_robust_tuned.png", dpi=110, bbox_inches="tight")
plt.close(fig)
print(f"Disimpan: {FIG_DIR / 'real_world_gradcam_09_noise_robust_tuned.png'}")
